# Joan Tryhard

### Imports

In [1]:
import pandas as pd
import sklearn
import imblearn
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix

### Get Data and Preprocess

In [2]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
import pandas as pd

# Load original data, keeping the 'language' column for filtering
train_orig = pd.read_csv("data/train_dataset_processed.csv")
test_orig = pd.read_csv("data/test_dataset_processed.csv")

# Get unique languages from training data
languages = train_orig['language'].unique()

# --- One-hot encode 'language' for the feature set ---
enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore') # Using sparse_output=False for easier DataFrame creation
# Fit encoder on training data languages
enc.fit(train_orig[['language']])

# Transform train data
language_encoded_train = enc.transform(train_orig[['language']])
language_df_train = pd.DataFrame(language_encoded_train,
                                 columns=enc.get_feature_names_out(['language']),
                                 index=train_orig.index)
train_processed = pd.concat([train_orig.drop(columns=['language']), language_df_train], axis=1)

# Transform test data
language_encoded_test = enc.transform(test_orig[['language']])
language_df_test = pd.DataFrame(language_encoded_test,
                                columns=enc.get_feature_names_out(['language']),
                                index=test_orig.index)
test_processed = pd.concat([test_orig.drop(columns=['language']), language_df_test], axis=1)

# Prepare full data and labels (these will be filtered per language or used for fallback)
X_full = train_processed.drop(columns=['root'])
y_full = train_processed['root']
X_test_processed = test_processed # This is the X_test to make predictions on

print("Data preprocessing complete.")
print(f"X_full shape: {X_full.shape}")
print(f"y_full shape: {y_full.shape}")
print(f"X_test_processed shape: {X_test_processed.shape}")

Data preprocessing complete.
X_full shape: (197479, 35)
y_full shape: (197479,)
X_test_processed shape: (194648, 35)


## Models

### Unimodel Random Forest

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold # Import GroupKFold
import numpy as np
import pandas as pd
from tqdm import tqdm
from imblearn.pipeline import Pipeline as ImbPipeline # Import ImbPipeline
from imblearn.over_sampling import SMOTE # Import SMOTE

# Assume X_full, y_full, train_orig, test_orig, X_test_processed are pre-defined
# and 'languages' list is available.
# For demonstration, let's create dummy versions if they don't exist.
if 'X_full' not in locals():
    print("Creating dummy data for demonstration...")
    n_train_samples = 1000
    n_test_samples = 200
    n_features = 10
    rng = np.random.RandomState(42)
    y_full_data_imbalanced = rng.choice([0, 1], size=n_train_samples, p=[0.9, 0.1])
    X_full_data = rng.rand(n_train_samples, n_features)
    
    # Create more structured sentence_ids for dummy data
    sentences_per_lang = 50
    nodes_per_sentence_avg = n_train_samples // (2 * sentences_per_lang) # for 2 languages
    
    train_sids = []
    current_sid = 0
    for _ in range(n_train_samples // nodes_per_sentence_avg):
        train_sids.extend([current_sid] * nodes_per_sentence_avg)
        current_sid += 1
    train_sids.extend([current_sid] * (n_train_samples - len(train_sids))) # fill remaining
    np.random.shuffle(train_sids)


    train_languages_list = ['English'] * (n_train_samples // 2) + ['Spanish'] * (n_train_samples // 2)
    if n_train_samples % 2 != 0: train_languages_list.append('English')
    np.random.shuffle(train_languages_list)

    X_full = pd.DataFrame(X_full_data, columns=[f'feature_{i}' for i in range(n_features)])
    y_full = pd.Series(y_full_data_imbalanced)
    
    train_orig_data = {
        'language': train_languages_list,
        'sentence_id': train_sids[:n_train_samples] # ensure correct length
    }
    train_orig = pd.DataFrame(train_orig_data)
    X_full.index = train_orig.index
    y_full.index = train_orig.index
    languages = train_orig['language'].unique()

    # Test data dummy
    test_sids = []
    current_sid_test = 0
    nodes_per_sentence_test_avg = n_test_samples // (2 * (sentences_per_lang // 5)) # fewer sentences in test
    for _ in range(n_test_samples // nodes_per_sentence_test_avg):
        test_sids.extend([current_sid_test] * nodes_per_sentence_test_avg)
        current_sid_test +=1
    test_sids.extend([current_sid_test] * (n_test_samples - len(test_sids)))
    np.random.shuffle(test_sids)

    X_test_data = rng.rand(n_test_samples, n_features)
    test_languages_list = ['English'] * (n_test_samples // 2) + ['Spanish'] * (n_test_samples // 2)
    if n_test_samples % 2 != 0: test_languages_list.append('Spanish')
    np.random.shuffle(test_languages_list)
    test_orig_data = {
        'language': test_languages_list,
        'sentence_id': test_sids[:n_test_samples]
    }
    test_orig = pd.DataFrame(test_orig_data)
    X_test_processed = pd.DataFrame(X_test_data, columns=[f'feature_{i}' for i in range(n_features)])
    X_test_processed.index = test_orig.index
    print(f"Dummy y_full class distribution:\\n{y_full.value_counts(normalize=True)}")
    # End dummy data creation


trained_models = {}
all_language_prob_predictions = []
lang_cols_to_drop = [col for col in X_full.columns if col.startswith('language_')]

# Updated param_grid for ImbPipeline
# You can expand this, e.g., by adding 'sampling__k_neighbors': [3, 5, 7] to tune SMOTE
# Keeping it simpler for this example to match original complexity more closely.
param_grid_pipeline = {
    # 'sampling__k_neighbors': [3, 5], # Example: To tune SMOTE k_neighbors
    'classification__n_estimators': [50, 500],      
    'classification__max_depth': [10, 100],     
    'classification__min_samples_split': [5, 10],    
    'classification__min_samples_leaf': [3, 5]
}


for lang in tqdm(languages, desc="Training models per language"):
    train_lang_indices = train_orig[train_orig['language'] == lang].index
    X_train_lang = X_full.loc[train_lang_indices].drop(columns=lang_cols_to_drop, errors='ignore')
    y_train_lang = y_full.loc[train_lang_indices]
    
    groups_lang = train_orig.loc[X_train_lang.index, 'sentence_id'].values

    if X_train_lang.empty or len(y_train_lang.unique()) < 2:
        print(f"Skipping language {lang}: Insufficient data or only one class.")
        continue
    
    print(f"\\nProcessing language: {lang} ({len(X_train_lang)} training samples)")
    print(f"  Class distribution for {lang} (before SMOTE):\\n{y_train_lang.value_counts(normalize=True).to_dict()}")

    n_unique_groups_lang = len(np.unique(groups_lang))
    cv_folds_lang = min(5, n_unique_groups_lang) 
    
    best_clf_lang = None

    # Check if number of samples in the smallest class is less than k_neighbors for SMOTE (default 5)
    # If so, SMOTE might fail or be ineffective. Fallback to simpler model without SMOTE or with adjusted SMOTE.
    min_class_count = y_train_lang.value_counts().min()
    can_smote = min_class_count >= 5 # Default k_neighbors for SMOTE is 5

    if cv_folds_lang < 2 or n_unique_groups_lang < cv_folds_lang or len(X_train_lang) < cv_folds_lang * 2 :
        print(f"  GroupKFold CV for {lang} skipped (n_unique_groups={n_unique_groups_lang}, cv_folds={cv_folds_lang}). Training with default-like parameters.")
        if can_smote:
            print("    Applying SMOTE before training.")
            sampler = SMOTE(random_state=42)
            # SMOTE requires at least k_neighbors samples in the minority class.
            # Adjust k_neighbors if min_class_count is too small.
            if min_class_count < sampler.k_neighbors:
                 current_k = max(1, min_class_count -1) # k must be < n_samples in minority class
                 print(f"    Adjusting SMOTE k_neighbors from {sampler.k_neighbors} to {current_k} for language {lang} due to small minority class size ({min_class_count}).")
                 sampler.k_neighbors = current_k

            if sampler.k_neighbors == 0: # Cannot run SMOTE if k is 0 (minority class has 1 sample)
                print(f"    Cannot apply SMOTE for {lang} as minority class has only {min_class_count} sample(s). Training without SMOTE.")
                X_train_lang_res, y_train_lang_res = X_train_lang, y_train_lang
            else:
                try:
                    X_train_lang_res, y_train_lang_res = sampler.fit_resample(X_train_lang, y_train_lang)
                    print(f"    Class distribution for {lang} (after SMOTE):\\n{pd.Series(y_train_lang_res).value_counts(normalize=True).to_dict()}")
                except ValueError as e_smote:
                    print(f"    Error during SMOTE for {lang}: {e_smote}. Training without SMOTE.")
                    X_train_lang_res, y_train_lang_res = X_train_lang, y_train_lang
        else:
            print(f"    Skipping SMOTE for {lang} as minority class has only {min_class_count} sample(s) (less than default k_neighbors=5). Training without SMOTE.")
            X_train_lang_res, y_train_lang_res = X_train_lang, y_train_lang
            
        best_clf_lang = RandomForestClassifier(random_state=42, n_estimators=100) # No class_weight
        best_clf_lang.fit(X_train_lang_res, y_train_lang_res)
    else:
        print(f"  Performing GridSearchCV for {lang} with GroupKFold (n_splits={cv_folds_lang}, scoring='roc_auc')...")
        group_kfold_lang = GroupKFold(n_splits=cv_folds_lang)
        
        # Create pipeline with SMOTE and RandomForestClassifier
        # SMOTE's k_neighbors must be less than the number of samples in the minority class within each fold.
        # This is tricky with GroupKFold. If SMOTE fails in a fold, GridSearchCV might error.
        # The try-except around fit should catch this.
        # For simplicity, we assume SMOTE will generally work or k_neighbors is adjusted if param_grid includes it.
        # If not tuning k_neighbors, ensure min_class_count >= SMOTE's default k_neighbors (5) for the whole lang dataset.
        
        smote_for_pipeline = SMOTE(random_state=42)
        if not can_smote:
             print(f"    Warning: Minority class for {lang} ({min_class_count} samples) is < default SMOTE k_neighbors (5). GridSearchCV might fail if SMOTE is used with default k_neighbors.")
             # Option: Don't use SMOTE in pipeline for this lang, or adjust k_neighbors if tuning it
             # For now, we proceed, relying on try-except or if k_neighbors is tuned.

        pipeline_estimator = ImbPipeline([
            ('sampling', smote_for_pipeline), 
            ('classification', RandomForestClassifier(random_state=42)) # No class_weight
        ])
        
        grid_search = GridSearchCV(estimator=pipeline_estimator, 
                                   param_grid=param_grid_pipeline, # Use pipeline-compatible param_grid
                                   cv=group_kfold_lang, 
                                   scoring='roc_auc', 
                                   n_jobs=-1,          
                                   verbose=0)        
        
        try:
            grid_search.fit(X_train_lang, y_train_lang, groups=groups_lang)
            best_clf_lang = grid_search.best_estimator_
            print(f"  Best params for {lang}: {grid_search.best_params_}")
            print(f"  Best CV score for {lang} (roc_auc): {grid_search.best_score_:.4f}")
            # You can also inspect the class distribution after SMOTE for the best model if needed,
            # but it's implicit in the pipeline.
        except Exception as e:
            print(f"  Error during GridSearchCV for {lang}: {e}. Training with default-like parameters (with SMOTE if possible).")
            if can_smote:
                print("    Applying SMOTE before training fallback.")
                sampler = SMOTE(random_state=42)
                if min_class_count < sampler.k_neighbors:
                    current_k = max(1, min_class_count-1)
                    sampler.k_neighbors = current_k
                if sampler.k_neighbors == 0:
                    X_train_lang_res, y_train_lang_res = X_train_lang, y_train_lang
                else:
                    try:
                        X_train_lang_res, y_train_lang_res = sampler.fit_resample(X_train_lang, y_train_lang)
                    except ValueError as e_smote_fallback:
                        print(f"    Error during SMOTE for {lang} (fallback): {e_smote_fallback}. Training without SMOTE.")
                        X_train_lang_res, y_train_lang_res = X_train_lang, y_train_lang
            else:
                X_train_lang_res, y_train_lang_res = X_train_lang, y_train_lang
            best_clf_lang = RandomForestClassifier(random_state=42, n_estimators=100)
            best_clf_lang.fit(X_train_lang_res, y_train_lang_res)

    trained_models[lang] = best_clf_lang
    print(f"  Model for {lang} trained.")

    test_lang_indices = test_orig[test_orig['language'] == lang].index
    if not test_lang_indices.empty:
        X_test_lang = X_test_processed.loc[test_lang_indices].drop(columns=lang_cols_to_drop, errors='ignore')
        if not X_test_lang.empty:
            prob_predictions_lang = best_clf_lang.predict_proba(X_test_lang)
            prob_dict = {}
            model_classes = list(best_clf_lang.classes_)
            if 0 in model_classes and 1 in model_classes:
                idx_0 = model_classes.index(0); idx_1 = model_classes.index(1)
                prob_dict[0] = prob_predictions_lang[:, idx_0]; prob_dict[1] = prob_predictions_lang[:, idx_1]
            elif 0 in model_classes:
                prob_dict[0] = prob_predictions_lang[:, model_classes.index(0)] if prob_predictions_lang.ndim > 1 else prob_predictions_lang
                prob_dict[1] = np.zeros_like(prob_dict[0])
            elif 1 in model_classes:
                prob_dict[1] = prob_predictions_lang[:, model_classes.index(1)] if prob_predictions_lang.ndim > 1 else prob_predictions_lang
                prob_dict[0] = np.zeros_like(prob_dict[1])
            else:
                print(f"  Warning: Model for {lang} (test pred) did not produce expected class probabilities. Classes: {model_classes}. Setting to 0.")
                prob_dict[0] = np.zeros(len(X_test_lang)); prob_dict[1] = np.zeros(len(X_test_lang))
            prob_predictions_lang_df = pd.DataFrame(prob_dict, index=X_test_lang.index)
            all_language_prob_predictions.append(prob_predictions_lang_df)
    else:
        print(f"  No test samples for language {lang}.")


# Combine all predictions
if all_language_prob_predictions:
    prob_predictions_df = pd.concat(all_language_prob_predictions).sort_index()
else:
    print("Warning: No language-specific predictions were made. Initializing empty prob_predictions_df for fallback.")
    prob_predictions_df = pd.DataFrame(0.0, index=X_test_processed.index, columns=[0, 1])

# Fallback logic
missing_indices = X_test_processed.index.difference(prob_predictions_df.index)
if not prob_predictions_df.empty:
   predicted_indices_for_fallback = prob_predictions_df[prob_predictions_df.sum(axis=1) == 0].index
   missing_indices = missing_indices.union(predicted_indices_for_fallback)

if not missing_indices.empty:
    print(f"\\nMissing or zero predictions for {len(missing_indices)} samples. Applying a fallback model.")
    print(f"  Fallback model class distribution (before SMOTE):\\n{y_full.value_counts(normalize=True).to_dict()}")

    groups_full = train_orig.loc[X_full.index, 'sentence_id'].values
    n_unique_groups_full = len(np.unique(groups_full))
    cv_folds_fallback = max(2, min(5, n_unique_groups_full))
    
    fallback_clf_final = None
    
    min_class_count_full = y_full.value_counts().min()
    can_smote_full = min_class_count_full >= 5 

    if cv_folds_fallback < 2 or n_unique_groups_full < cv_folds_fallback or len(X_full) < cv_folds_fallback * 2:
        print(f"  GroupKFold CV for fallback model skipped. Training with default-like parameters.")
        if can_smote_full:
            print("    Applying SMOTE to full dataset for fallback.")
            sampler_full = SMOTE(random_state=42)
            if min_class_count_full < sampler_full.k_neighbors:
                current_k_full = max(1, min_class_count_full -1)
                sampler_full.k_neighbors = current_k_full
            
            if sampler_full.k_neighbors == 0:
                X_full_res, y_full_res = X_full, y_full
            else:
                try:
                    X_full_res, y_full_res = sampler_full.fit_resample(X_full, y_full)
                    print(f"    Fallback class distribution (after SMOTE):\\n{pd.Series(y_full_res).value_counts(normalize=True).to_dict()}")
                except ValueError as e_smote_full_fallback:
                    print(f"    Error during SMOTE for fallback (full data): {e_smote_full_fallback}. Training without SMOTE.")
                    X_full_res, y_full_res = X_full, y_full
        else:
            print(f"    Skipping SMOTE for fallback (full data) as minority class has only {min_class_count_full} sample(s). Training without SMOTE.")
            X_full_res, y_full_res = X_full, y_full
            
        fallback_clf_final = RandomForestClassifier(random_state=42, n_estimators=100) # No class_weight
        fallback_clf_final.fit(X_full_res, y_full_res)
    else:
        print(f"  Performing GridSearchCV for fallback model with GroupKFold (n_splits={cv_folds_fallback}, scoring='roc_auc')...")
        group_kfold_fallback = GroupKFold(n_splits=cv_folds_fallback)

        smote_for_fallback_pipeline = SMOTE(random_state=42)
        if not can_smote_full:
            print(f"    Warning: Minority class for full data ({min_class_count_full} samples) is < default SMOTE k_neighbors (5). GridSearchCV for fallback might fail.")

        fallback_pipeline_estimator = ImbPipeline([
            ('sampling', smote_for_fallback_pipeline),
            ('classification', RandomForestClassifier(random_state=42)) # No class_weight
        ])
        
        fallback_grid_search = GridSearchCV(estimator=fallback_pipeline_estimator,
                                            param_grid=param_grid_pipeline, 
                                            cv=group_kfold_fallback,
                                            scoring='roc_auc',
                                            n_jobs=-1,
                                            verbose=0)
        try:
            fallback_grid_search.fit(X_full, y_full, groups=groups_full)
            fallback_clf_final = fallback_grid_search.best_estimator_
            print(f"  Best params for fallback model: {fallback_grid_search.best_params_}")
            print(f"  Best CV score for fallback model (roc_auc): {fallback_grid_search.best_score_:.4f}")
        except Exception as e:
            print(f"  Error during GridSearchCV for fallback model: {e}. Training with default-like parameters (with SMOTE if possible).")
            if can_smote_full:
                print("    Applying SMOTE to full dataset for fallback (after error).")
                sampler_full = SMOTE(random_state=42)
                if min_class_count_full < sampler_full.k_neighbors:
                    current_k_full = max(1, min_class_count_full-1)
                    sampler_full.k_neighbors = current_k_full
                
                if sampler_full.k_neighbors == 0:
                     X_full_res, y_full_res = X_full, y_full
                else:
                    try:
                        X_full_res, y_full_res = sampler_full.fit_resample(X_full, y_full)
                    except ValueError as e_smote_full_error_fallback:
                        print(f"    Error during SMOTE for fallback (full data, after error): {e_smote_full_error_fallback}. Training without SMOTE.")
                        X_full_res, y_full_res = X_full, y_full
            else:
                 X_full_res, y_full_res = X_full, y_full

            fallback_clf_final = RandomForestClassifier(random_state=42, n_estimators=100)
            fallback_clf_final.fit(X_full_res, y_full_res)

    X_test_missing = X_test_processed.loc[missing_indices]
    if not X_test_missing.empty:
        prob_predictions_missing = fallback_clf_final.predict_proba(X_test_missing)
        prob_dict_fallback = {}
        fallback_model_classes = list(fallback_clf_final.classes_)
        if 0 in fallback_model_classes and 1 in fallback_model_classes:
            idx_0_fb = fallback_model_classes.index(0); idx_1_fb = fallback_model_classes.index(1)
            prob_dict_fallback[0] = prob_predictions_missing[:, idx_0_fb]; prob_dict_fallback[1] = prob_predictions_missing[:, idx_1_fb]
        elif 0 in fallback_model_classes:
            prob_dict_fallback[0] = prob_predictions_missing[:, fallback_model_classes.index(0)] if prob_predictions_missing.ndim > 1 else prob_predictions_missing
            prob_dict_fallback[1] = np.zeros_like(prob_dict_fallback[0])
        elif 1 in fallback_model_classes:
            prob_dict_fallback[1] = prob_predictions_missing[:, fallback_model_classes.index(1)] if prob_predictions_missing.ndim > 1 else prob_predictions_missing
            prob_dict_fallback[0] = np.zeros_like(prob_dict_fallback[1])
        else:
            print("  Warning: Fallback model (test pred) did not produce expected class probabilities. Setting to 0.")
            prob_dict_fallback[0] = np.zeros(len(X_test_missing)); prob_dict_fallback[1] = np.zeros(len(X_test_missing))
        prob_predictions_missing_df = pd.DataFrame(prob_dict_fallback, index=missing_indices)
        
        # Safely update or concat
        if prob_predictions_df.empty:
            prob_predictions_df = prob_predictions_missing_df
        else:
            # Update existing rows, then concat new ones
            prob_predictions_df.update(prob_predictions_missing_df)
            newly_added_indices = prob_predictions_missing_df.index.difference(prob_predictions_df.index)
            if not newly_added_indices.empty:
                prob_predictions_df = pd.concat([prob_predictions_df, prob_predictions_missing_df.loc[newly_added_indices]])
        
        prob_predictions_df = prob_predictions_df.sort_index()
        print("  Fallback predictions applied.")

# Final checks and re-alignment
if len(prob_predictions_df) != len(X_test_processed):
    print(f"Warning: Final prediction count ({len(prob_predictions_df)}) does not match test set size ({len(X_test_processed)}). Re-aligning.")
    prob_predictions_df = prob_predictions_df.reindex(X_test_processed.index)
    if prob_predictions_df.isnull().values.any():
         prob_predictions_df.fillna(0.5, inplace=True) 
         print("Filled NaNs introduced by re-alignment with 0.5/0.5.")

prob_predictions_df.columns = [0, 1] 

print("\\nFinal prob_predictions_df shape:", prob_predictions_df.shape)
if not prob_predictions_df.empty:
    print("Sample of final predictions:")
    print(prob_predictions_df.head())
else:
    print("prob_predictions_df is empty after all processing.")

Training models per language:   0%|          | 0/21 [00:00<?, ?it/s]

\nProcessing language: Japanese (12906 training samples)
  Class distribution for Japanese (before SMOTE):\n{0: 0.9612583294591662, 1: 0.038741670540833724}
  Performing GridSearchCV for Japanese with GroupKFold (n_splits=5, scoring='roc_auc')...
  Best params for Japanese: {'classification__max_depth': 10, 'classification__min_samples_leaf': 3, 'classification__min_samples_split': 10, 'classification__n_estimators': 500}
  Best CV score for Japanese (roc_auc): 0.6925
  Model for Japanese trained.


Training models per language:   5%|▍         | 1/21 [03:26<1:08:59, 206.99s/it]

\nProcessing language: Finnish (6786 training samples)
  Class distribution for Finnish (before SMOTE):\n{0: 0.9263188918361333, 1: 0.07368110816386679}
  Performing GridSearchCV for Finnish with GroupKFold (n_splits=5, scoring='roc_auc')...
  Best params for Finnish: {'classification__max_depth': 10, 'classification__min_samples_leaf': 3, 'classification__min_samples_split': 10, 'classification__n_estimators': 500}
  Best CV score for Finnish (roc_auc): 0.8470
  Model for Finnish trained.


Training models per language:  10%|▉         | 2/21 [05:24<48:52, 154.32s/it]  

...

\nProcessing language: Arabic (9243 training samples)
  Class distribution for Arabic (before SMOTE):\n{0: 0.9459050091961484, 1: 0.054094990803851564}
  Performing GridSearchCV for Arabic with GroupKFold (n_splits=5, scoring='roc_auc')...
  Best params for Arabic: {'classification__max_depth': 10, 'classification__min_samples_leaf': 5, 'classification__min_samples_split': 5, 'classification__n_estimators': 500}
  Best CV score for Arabic (roc_auc): 0.8289
  Model for Arabic trained.


Training models per language:  95%|█████████▌| 20/21 [37:31<01:26, 86.16s/it]

\nProcessing language: Russian (8224 training samples)
  Class distribution for Russian (before SMOTE):\n{0: 0.9392023346303502, 1: 0.060797665369649805}
  Performing GridSearchCV for Russian with GroupKFold (n_splits=5, scoring='roc_auc')...


Training models per language: 100%|██████████| 21/21 [38:47<00:00, 110.83s/it]

  Best params for Russian: {'classification__max_depth': 10, 'classification__min_samples_leaf': 5, 'classification__min_samples_split': 5, 'classification__n_estimators': 500}
  Best CV score for Russian (roc_auc): 0.8669
  Model for Russian trained.
\nFinal prob_predictions_df shape: (194648, 2)
Sample of final predictions:
          0         1
0  0.947874  0.052126
1  0.921597  0.078403
2  0.939418  0.060582
3  0.719180  0.280820
4  0.950277  0.049723


## Evaluating results

In [4]:
preds = pd.DataFrame({
    'language': test_orig['language'], # Get language from original test data
    'sentence_id': test_orig['sentence_id'],
    'node': test_orig['node'],
    'zero': prob_predictions_df[0],
    'root_prob': prob_predictions_df[1],
}, index=test_orig.index)

In [5]:
# Find the index of the row with max 'root_prob' for each group
idx_max_prob_per_group = preds.groupby(['language', 'sentence_id'], sort=False)['root_prob'].idxmax()

# Select these rows from the 'preds' DataFrame
preds_at_max_prob = preds.loc[idx_max_prob_per_group]

# Create the 'preds_grouped' DataFrame by selecting and renaming the 'node' column
preds_grouped = preds_at_max_prob[['node']].copy()
preds_grouped.rename(columns={'node': 'root'}, inplace=True)

# Reset index to be sequential (0, 1, 2, ...)
preds_grouped.reset_index(drop=True, inplace=True)

# Add 'id' column (1-based index for submission)
preds_grouped['id'] = range(1, len(preds_grouped) + 1)

# Ensure columns are in the order ['id', 'root']
preds_grouped = preds_grouped[['id', 'root']]

# Save the predictions to a CSV file
preds_grouped.to_csv('data/predictions_submission_multimodel.csv', index=False)


In [6]:
current_predictions = pd.read_csv('data/predictions_submission_multimodel.csv')
print("Loaded submission file (head):")
display(current_predictions.head())

Loaded submission file (head):


,id,root
0,1,4
1,2,17
2,3,6
3,4,14
4,5,4


In [7]:
try:
    kaggle_perfect_predictions = pd.read_csv("data/kaggle_perfect_predictions.csv")
    
    if len(kaggle_perfect_predictions) == len(current_predictions):
        y_true_eval = kaggle_perfect_predictions['root']
        y_pred_eval = current_predictions['root']
        # Ensure labels are consistent for classification_report if some node IDs are missing in either set
        all_labels = sorted(list(set(y_true_eval) | set(y_pred_eval)))
        print(classification_report(y_true_eval, y_pred_eval, labels=all_labels, zero_division=0))
    else:
        print("Skipping classification_report: Row count mismatch between perfect predictions and current predictions.")
        print(f"Kaggle perfect: {len(kaggle_perfect_predictions)}, Current: {len(current_predictions)}")
except FileNotFoundError:
    print("Kaggle perfect predictions file not found. Skipping classification report.")
except Exception as e:
    print(f"An error occurred during classification report generation: {e}")

              precision    recall  f1-score   support

           1       0.34      0.34      0.34       690
           2       0.34      0.35      0.34       675
           3       0.33      0.36      0.34       687
           4       0.28      0.30      0.29       641
           5       0.34      0.35      0.35       693
           6       0.34      0.36      0.35       626
           7       0.31      0.32      0.32       653
           8       0.31      0.34      0.32       607
           9       0.30      0.33      0.32       547
          10       0.28      0.29      0.29       510
          11       0.30      0.29      0.29       491
          12       0.28      0.30      0.29       407
          13       0.27      0.26      0.27       390
          14       0.28      0.30      0.29       346
          15       0.26      0.25      0.25       336
          16       0.27      0.23      0.25       312
          17       0.26      0.24      0.25       248
          18       0.26    

In [8]:
def evaluate_model(y_true_df, y_pred_df):
    # Assuming y_true_df and y_pred_df are DataFrames with 'root' and matching length
    if 'root' not in y_true_df.columns or 'root' not in y_pred_df.columns:
        print("Error: 'root' column missing in one of the dataframes.")
        return
    if len(y_true_df) != len(y_pred_df):
        print("Error: Length mismatch between true and predicted values.")
        print(f"  Length of true values: {len(y_true_df)}")
        print(f"  Length of predicted values: {len(y_pred_df)}")
        return
        
    y_true = y_true_df['root']
    y_pred = y_pred_df['root']
    correct_predictions = (y_true == y_pred).sum()
    total_predictions = len(y_true)
    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    print(f"Number of correct predictions: {correct_predictions} / {total_predictions}")
    print(f"Evaluation accuracy: {accuracy:.4f}")

try:
    # Reload for this function to ensure clean state
    kaggle_perfect_predictions_eval = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions_eval = pd.read_csv('data/predictions_submission_multimodel.csv')
    evaluate_model(kaggle_perfect_predictions_eval, current_predictions_eval)
except FileNotFoundError:
    print("One of the prediction files not found. Skipping custom evaluation.")
except Exception as e:
    print(f"An error occurred during custom evaluation: {e}")

Number of correct predictions: 3128 / 10395
Evaluation accuracy: 0.3009
